In [ ]:
# imports

import math

import numpy as np


# Kinematyka napędu i odometria

## Kinematyka napędu

Robot ma trzy **koła omnikierunkowe (omni wheels)** rozmieszczone co 120° w odległości $L = 0{,}132$ m od środka. Na obwodzie każdego koła znajdują się bierne rolki. Koło napędza więc robota wyłącznie w kierunku stycznym do okręgu, na którym leżą koła, a w kierunku prostopadłym przetacza się swobodnie na rolkach. Dzięki temu robot może poruszać się w dowolnym kierunku i jednocześnie obracać wokół własnej osi; taki napęd nazywamy **holonomicznym (holonomic)**.

![Koło omnikierunkowe: ruch w kierunku napędu, a następnie swobodny ruch na rolkach. Model koła kiddo_omniwheel (makerspet, Apache 2.0)](img/omni_wheel_rolling.gif)

![Rozmieszczenie kół robota i kierunki, w których każde z nich napędza platformę](img/wheel_forces_top.png)

Koło ustawione pod kątem $\beta_i$ napędza robota w kierunku $(-\sin\beta_i, \cos\beta_i)$, zatem jego obwód porusza się z prędkością

$$
w_i = -\sin(\beta_i)\, v_x + \cos(\beta_i)\, v_y + L\, \omega,
$$

gdzie $(v_x, v_y)$ jest prędkością robota w jego własnym układzie (oś $x$ do przodu, oś $y$ w lewo), a $\omega$ prędkością obrotu. W naszym robocie $\beta_1 = 60°$, $\beta_2 = 180°$, $\beta_3 = 300°$. Trzy równania tworzą układ $\mathbf{w} = \mathbf{M}\mathbf{v}$, gdzie wiersz $i$ macierzy $\mathbf{M}$ to $[-\sin\beta_i,\ \cos\beta_i,\ L]$. Biblioteka korzysta z niego, zamieniając polecenie `set_vector` na prędkości kół.

Odometria działa w odwrotną stronę. Jeżeli od poprzedniego kroku obwody kół przebyły drogi $\Delta s_1, \Delta s_2, \Delta s_3$, to rozwiązanie układu $\mathbf{M}\,[\Delta x, \Delta y, \Delta\theta]^T = [\Delta s_1, \Delta s_2, \Delta s_3]^T$ jest przesunięciem w układzie robota. Do układu pudła przesunięcie obraca się o kąt robota w połowie kroku, $\bar\theta = \theta + \Delta\theta / 2$:

$$
x \leftarrow x + \Delta x \cos\bar\theta - \Delta y \sin\bar\theta, \qquad
y \leftarrow y + \Delta x \sin\bar\theta + \Delta y \cos\bar\theta, \qquad
\theta \leftarrow \theta + \Delta\theta
$$

W ten sam sposób, 100 razy na sekundę, odometrię wyznacza sterownik robota.

**Zadanie 1**

1. Zbuduj macierz `M` (3 × 3) dla kątów zapisanych w `WHEEL_ANGLES`. Wiersz $i$ to $[-\sin\beta_i,\ \cos\beta_i,\ L]$, więc kolumny macierzy to kolejno: `-np.sin(WHEEL_ANGLES)`, `np.cos(WHEEL_ANGLES)` i trzy wartości `L`. Funkcje `np.sin` i `np.cos` przyjmują argument w radianach.
2. Uzupełnij funkcję `odometry_step(pose, ds)`. Funkcja przyjmuje **pozę (pose)**, czyli tablicę `[x, y, yaw]`, oraz przesunięcia obwodów kół `ds` (trzy wartości w metrach) i zwraca nową pozę:
   1. wyznacz przesunięcie w układzie robota $[\Delta x, \Delta y, \Delta\theta]$, rozwiązując układ $\mathbf{M}\,[\Delta x, \Delta y, \Delta\theta]^T = \Delta\mathbf{s}$ funkcją [`np.linalg.solve`](https://numpy.org/doc/stable/reference/generated/numpy.linalg.solve.html)`(M, ds)`,
   2. oblicz kąt w połowie kroku $\bar\theta = \theta + \Delta\theta / 2$,
   3. obróć przesunięcie o $\bar\theta$ i dodaj je do $x$, $y$, a $\Delta\theta$ dodaj do kąta (wzory powyżej),
   4. zwróć nową pozę jako `np.array([x, y, yaw])`.

Komórka pod zadaniem sprawdza rozwiązanie na przejeździe 1 m na wprost w krokach po 2 mm. Przy idealnych kołach odometria powinna wskazać $(1, 0, 0°)$. Następnie przejazd jest powtarzany dla koła 1 o 2% większego, niż zakłada model.

In [ ]:
L = 0.132
WHEEL_ANGLES = np.radians([60, 180, 300])

M = ...


def odometry_step(pose, ds):
    x, y, yaw = pose
    ...

In [ ]:
def drive_straight(wheel_scale):
    pose = np.zeros(3)
    for _ in range(500):
        pose = odometry_step(pose, wheel_scale * (M @ [0.002, 0.0, 0.0]))
    return pose


for name, scale in (
    ("exact wheels", [1.0, 1.0, 1.0]),
    ("wheel 1 +2%", [1.02, 1.0, 1.0]),
):
    x, y, yaw = drive_straight(np.array(scale))
    print(
        f"{name:14s} x = {x:.3f} m, y = {y * 100:+.1f} cm, yaw = {math.degrees(yaw):+.1f}°"
    )

TypeError: unsupported operand type(s) for @: 'ellipsis' and 'list'

Różnica 2% w promieniu jednego koła powoduje po przejechaniu metra odchylenie o kilka centymetrów w bok i kilka stopni obrotu, i to w modelu pozbawionym poślizgu. Odometria całkuje każdy taki błąd i nie ma mechanizmu, który by go korygował.


# ważne

Oba czujniki mierzą odległość od swojej obudowy, a LiDAR od swojego środka. Nagranie poniżej zaczyna się od sekundy postoju, w której zmierzymy różnicę między odczytami czujników a LiDAR-u i dodamy ją do czujników. W ten sposób usuwamy błąd systematyczny i wszystkie trzy źródła mierzą tę samą wielkość: odległość $d$ od LiDAR-u do przedniej ściany.

DO ZMIANY

# Robot omni

## Wstęp

Aby robot mógł dojechać do zadanego punktu, musi znać własne położenie. Żaden z czujników robota nie mierzy go jednak bezpośrednio i bez błędu. Na laboratorium porównamy kilka źródeł informacji o położeniu robota omnikierunkowego jeżdżącego w pudle 1 × 1 m:

* **odometria (odometry)** - całkowanie obrotów kół; estymata jest gładka, ale jej błąd rośnie wraz z przejechaną drogą,
* **LiDAR** - odległości do ścian pudła; nie dryfuje, ale jest zaszumiony,
* **czujniki odległości (distance sensors)**, laserowy ToF i ultradźwiękowy - tanie, ale mierzą odległość tylko w jednym kierunku,
* **filtr Kalmana (Kalman filter)** - łączy gładką odometrię z pomiarem odległości, który nie dryfuje.

Laboratorium składa się z dwóch części. W pierwszej wyprowadzimy kinematykę napędu, wyznaczymy położenie z LiDAR-u i zmierzymy dryf odometrii podczas jazdy i obrotu. W drugiej wyznaczymy z pomiarów parametry gotowego filtru Kalmana w jednym wymiarze, porównamy, jak sprawdza się z LiDAR-em, czujnikiem ToF i czujnikiem ultradźwiękowym, a na koniec zatrzymamy robota przed przeszkodą.

Wykorzystywane biblioteki: NumPy i pandas do obliczeń, Bokeh do wykresów oraz `omni_drive`, biblioteka sterująca robotem.

**Bezpieczeństwo.** Przed uruchomieniem komórki, która porusza robotem, należy ustawić go na środku pudła, przodem do ściany. Ruch należy zlecać metodą `await robot.drive_for(...)`, która zatrzymuje robota po zadanym czasie, również w przypadku przerwania komórki (■). Metoda `robot.set_vector(...)` utrzymuje zadany ruch aż do kolejnego polecenia, nawet po zakończeniu komórki. Robota można w każdej chwili zatrzymać wywołaniem `robot.stop()`.

## Połączenie z robotem

Notebook działa na komputerze robota (Raspberry Pi), z którym łączymy się przez sieć Wi-Fi. Gotowość robota sygnalizuje zielona dioda Raspberry Pi migająca w rytmie bicia serca.

`OmniDrive()` nawiązuje połączenie ze sterownikiem kół (ESP32), który 50 razy na sekundę przesyła odometrię i odczyty czujników odległości, oraz z LiDAR-em, który przesyła skan 10 razy na sekundę. `robot.print_status()` wypisuje stan baterii, odometrię, prędkości kół i odczyty czujników. Jeśli komórka zgłosi brak telemetrii, należy odczekać kilka sekund i uruchomić ją ponownie.

Robot ma trzy czujniki ToF (`robot.ir_1`, `ir_2`, `ir_3`: lewy, środkowy, prawy) i dwa ultradźwiękowe (`robot.us_1` lewy, `robot.us_2` środkowy; prawy nie jest podłączony i wyświetla się jako `---`). Na laboratorium korzystamy tylko z czujników środkowych, skierowanych do przodu.

Kierunki ruchu podawane są względem pudła: kąt `0°` oznacza kierunek, w którym robot był zwrócony przodem w chwili wywołania `robot.reset_pose()`. Podczas jazdy bez obrotu biblioteka utrzymuje orientację robota regulatorem, więc robot nie skręca, nawet gdy koła się ślizgają.

In [7]:
robot = OmniDrive()
await asyncio.sleep(1.0)
robot.reset_pose()
robot.print_status()

CancelledError: 

### Funkcje pomocnicze

Poza sterowaniem kołami biblioteka `omni_drive` udostępnia funkcje używane w dalszej części notebooka:

* `robot.stop()` zatrzymuje robota,
* `await robot.drive_for(duration, vector)` jedzie z zadanym wektorem przez czas `duration` (obiekt `Time`, np. `Time.s(...)`) i zatrzymuje robota, również po przerwaniu komórki; kończy się, gdy robot już stoi,
* `robot.pose` to położenie z odometrii względem chwili ostatniego `robot.reset_pose()`: `robot.pose.x.meters`, `robot.pose.y.meters` i `robot.pose.yaw.radians`,
* `sensor.distance.meters`, np. `robot.ir_2.distance.meters`, to odczyt czujnika odległości w metrach albo `inf`, gdy czujnik nie zmierzył odległości,
* `robot.lidar.point_cloud` zwraca punkty ostatniego skanu LiDAR-u jako tablicę N × 2 w metrach, w układzie robota (oś $x$ do przodu, oś $y$ w lewo), bez odbić od obudowy robota,
* `Clock()` to zegar czasu rzeczywistego synchronizujący pętlę sterowania (np. `async for t in Clock():`),
* `Recorder(nazwa=funkcja, ...)` co 50 ms zapisuje wartości podanych funkcji; `start()` rozpoczyna nagrywanie, a `stop()` je kończy i zwraca tabelę pandas z kolumną `t` (czas w sekundach),
* `plot_series(t, series, title, y_label, dots=())` z modułu `omni_drive.ui` rysuje przebiegi czasowe ze słownika `series`; serie wymienione w `dots` rysowane są punktami.

Przy poleceniu `velocity = 1.0` obwód koła porusza się z prędkością $s_{max} \approx 0{,}39$ m/s (`S_MAX`), co wynika z parametrów enkoderów: maksymalnie 175 impulsów na 100 ms, 990 impulsów na obrót, promień koła 35 mm.

In [ ]:
S_MAX = 0.39


### Obrót

Dotychczas robot poruszał się bez zmiany orientacji. Trzecia składowa `DriveVector`, `omega`, steruje obrotem niezależnie od ruchu postępowego. Wartość `omega = 1.0` oznacza, że wszystkie koła obracają się z pełną prędkością w tym samym kierunku; robot obraca się wtedy z prędkością $\omega_{max} = s_{max} / L$.

![Jazda na wprost, obrót w miejscu i jazda z obrotem. Prędkości kół wynikają z kinematyki odwrotnej](img/kiwi_motion.gif)

Podczas obrotu poślizg kół jest większy niż podczas ruchu postępowego, a odometria tego nie widzi. Kąt robota zmierzymy więc również LiDAR-em: funkcja `wall_angle()` w komórce pod zadaniem dopasowuje prostą do punktów przedniej ściany i zwraca kąt, o jaki robot jest obrócony względem niej.

**Zadanie 3**

1. Wyznacz `OMEGA_MAX`, czyli prędkość obrotu w rad/s przy `omega = 1.0`, na podstawie `S_MAX` i `L` (wzór na $\omega_{max}$ powyżej).
2. Uzupełnij funkcję `rotate(turns, omega=0.3)`, która obraca robota w miejscu o `turns` pełnych obrotów (wartość ujemna oznacza obrót w prawo):
   1. kąt do obrócenia to `abs(turns) * 2 * math.pi` radianów,
   2. przy poleceniu `omega` robot obraca się z prędkością `abs(omega) * OMEGA_MAX` rad/s; czas obrotu to kąt podzielony przez tę prędkość,
   3. kierunek obrotu wynika ze znaku `turns`: użyj `math.copysign(omega, turns)` jako prędkości obrotu (czas musi być dodatni),
   4. zleć ruch: `await robot.drive_for(Time.s(czas), DriveVector(velocity=0.0, heading=Angle(0), omega=prędkość_obrotu))`.

Komórka pod zadaniem obraca robota o jeden pełny obrót w miejscu i porównuje, o ile kąt robota po obrocie różni się od kąta przed obrotem według odometrii i według LiDAR-u. Przed uruchomieniem ustaw robota na środku pudła, przodem do ściany.

In [ ]:
OMEGA_MAX = ...


async def rotate(turns, omega=0.3): ...

In [ ]:
def wall_angle(points):
    ahead = points[(points[:, 0] > 0) & (np.abs(points[:, 1]) < 0.25)]
    slope = np.polyfit(ahead[:, 1], ahead[:, 0], 1)[0]
    return math.atan(slope)


await asyncio.sleep(0.5)
odometry_before, lidar_before = (
    robot.pose.yaw.radians,
    wall_angle(robot.lidar.point_cloud),
)
await rotate(1)
await asyncio.sleep(0.5)
odometry_after, lidar_after = (
    robot.pose.yaw.radians,
    wall_angle(robot.lidar.point_cloud),
)
print(f"OMEGA_MAX = {OMEGA_MAX:.2f} rad/s")
print(
    f"odometry residual after a full turn: {math.degrees(wrap_angle(odometry_after - odometry_before)):+.1f}°"
)
print(
    f"LiDAR residual after a full turn:    {math.degrees(lidar_after - lidar_before):+.1f}°"
)

Odometria uważa, że robot wykonał niemal dokładnie pełny obrót, a LiDAR pokazuje różnicę wynikającą z poślizgu kół. Po takim obrocie odometria myli się co do kierunku jazdy, więc każdy kolejny przejazd zwiększa błąd położenia.


### Czujniki ToF i ultradźwiękowe

Czujnik **ToF (time of flight)** VL53L1X wysyła impuls światła podczerwonego i mierzy czas powrotu odbitego światła. Mierzy odległość do ok. 4 m w wąskim stożku (ok. 27°), co 50 ms, z dokładnością kilku milimetrów.

Czujnik **ultradźwiękowy (ultrasonic)** HC-SR04 wysyła impuls o częstotliwości 40 kHz i mierzy czas $t$ do powrotu echa. Dźwięk pokonuje drogę do przeszkody dwukrotnie, zatem odległość wynosi $d = c\,t/2$, gdzie $c \approx 343$ m/s. Sterownik robota ogranicza zasięg do 2 m; gdy echo nie wróci, `sensor.distance` zwraca `Distance.inf`.

![HC-SR04: impuls 40 kHz biegnie do ściany i wraca, a pin ECHO pozostaje w stanie wysokim przez czas przelotu](img/ultrasonic_echo.gif)

Fala odbita od ściany ustawionej pod kątem może nie powrócić do czujnika; wówczas czujnik nie zwraca odczytu albo zwraca echo od innego obiektu.

![Pomiar czasu przelotu: echo od ściany prostopadłej i od ściany pod kątem](img/ultrasonic.png)



Błędy pomiaru dzielą się na trzy rodzaje, z których każdy wymaga innego postępowania. **Błąd systematyczny (bias)** usuwa się kalibracją, **szum losowy (random noise)** uśrednia filtr, natomiast **pomiary odstające (outliers)** należy wykryć i odrzucić.

![Trzy rodzaje błędów pomiaru](img/sensor_errors.png)
